In [ ]:
!pip install psycopg2-binary SQLAlchemy dagshub mlflow optuna xgboost scikit-learn tqdm PyDrive pyarrow fastparquet

import os
import gc
import shutil
import glob
import pandas as pd
import numpy as np
import optuna
import mlflow
import dagshub
import joblib
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sqlalchemy import create_engine, text
from kaggle_secrets import UserSecretsClient
from tqdm.auto import tqdm

# 1. OTORISASI GOOGLE DRIVE
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive

print("Mencari dan mempersiapkan KTP Google Drive...")
pencarian_file = glob.glob('/kaggle/input/**/client_secrets.json', recursive=True)

if pencarian_file:
    sumber_json = pencarian_file[0]
    tujuan_json = '/kaggle/working/client_secrets.json'
    shutil.copy(sumber_json, tujuan_json)
    print(f"File berhasil ditemukan dan diamankan di ruang kerja!")
else:
    print("[ERROR] File client_secrets.json tidak ada di panel Input!")

print("Meminta akses Google Drive...")
gauth = GoogleAuth()
gauth.CommandLineAuth() 
drive = GoogleDrive(gauth)
print("Akses Google Drive diamankan!")

import warnings
warnings.filterwarnings('ignore')

# 2. KONFIGURASI DAGSHUB
user_secrets = UserSecretsClient()
repo_owner = user_secrets.get_secret("DAGSHUB_REPO_OWNER")
repo_name = user_secrets.get_secret("DAGSHUB_REPO_NAME")
tracking_uri = user_secrets.get_secret("MLFLOW_TRACKING_URI")
db_url_pandas = user_secrets.get_secret("DATABASE_URL") 

if repo_owner and repo_name:
    print(f"Menghubungkan ke DagsHub ({repo_owner}/{repo_name})...")
    dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)
else:
    mlflow.set_tracking_uri(tracking_uri)

EXPERIMENT_NAME = "AeroGuard_TimeSeries_Mentok_Kanan"
mlflow.set_experiment(EXPERIMENT_NAME)

daftar_polutan = ['PM25', 'PM10', 'SO2', 'CO', 'NO2', 'O3']

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.4/987.4 kB 30.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 92.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 103.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 90.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.4 MB/s et

Enter verification code:  4/1AeoWuM-_icKiHad-DxzRuuM1-5lMtpxBeRMFhZuPOZfyksKZj-9SIQgMJ84


Authentication successful.
Akses Google Drive diamankan!
Menghubungkan ke DagsHub (tegarkusuma12/Web-ISPU)...


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=6f787e24-7f20-4946-8950-61049d123421&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=5e856d6ac16d5fd52b0a802973f2777cffcd7f14c0bf8462ef5559eaa7829f90




Accessing as YonoBengkel

Initialized MLflow to track repo "tegarkusuma12/Web-ISPU"

Repository tegarkusuma12/Web-ISPU initialized!

In [ ]:
print("Membuka jembatan ke Supabase...")
engine = create_engine(db_url_pandas)
RENTANG_WAKTU = "2 years"

query_sql = f"""
    SELECT waktu_aktual, id_wilayah, pm25, pm10, so2, co, no2, ozon
    FROM public.data_historis
    WHERE waktu_aktual >= NOW() - INTERVAL '{RENTANG_WAKTU}';
"""

ukuran_chunk = 50000
chunks = []

with engine.connect() as conn:
    conn.execute(text("SET statement_timeout = 0"))
    conn.commit() 
    stream_conn = conn.execution_options(stream_results=True)
    for i, chunk in enumerate(pd.read_sql(text(query_sql), stream_conn, chunksize=ukuran_chunk)):
        print(f"Menyedot batch ke-{i+1}...")
        chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)

df.rename(columns={
    'id_wilayah': 'nama_wilayah', 
    'pm25': 'PM25', 'pm10': 'PM10', 'so2': 'SO2', 
    'co': 'CO', 'no2': 'NO2', 'ozon': 'O3'
}, inplace=True)

pemetaan_id_kota = {
    1: 'Kota Surabaya', 2: 'Kota Malang', 3: 'Kabupaten Malang', 4: 'Kota Batu', 
    5: 'Kabupaten Sidoarjo', 6: 'Kabupaten Gresik', 7: 'Kabupaten Bangkalan', 
    8: 'Kabupaten Sampang', 9: 'Kabupaten Pamekasan', 10: 'Kabupaten Sumenep', 
    11: 'Kota Mojokerto', 12: 'Kabupaten Mojokerto', 13: 'Kabupaten Jombang', 
    14: 'Kabupaten Bojonegoro', 15: 'Kabupaten Tuban', 16: 'Kabupaten Lamongan', 
    17: 'Kota Madiun', 18: 'Kabupaten Madiun', 19: 'Kabupaten Ngawi', 
    20: 'Kabupaten Magetan', 21: 'Kabupaten Ponorogo', 22: 'Kabupaten Pacitan', 
    23: 'Kota Kediri', 24: 'Kabupaten Kediri', 25: 'Kabupaten Nganjuk', 
    26: 'Kota Blitar', 27: 'Kabupaten Blitar', 28: 'Kabupaten Tulungagung', 
    29: 'Kabupaten Trenggalek', 30: 'Kota Pasuruan', 31: 'Kabupaten Pasuruan', 
    32: 'Kota Probolinggo', 33: 'Kabupaten Probolinggo', 34: 'Kabupaten Lumajang', 
    35: 'Kabupaten Jember', 36: 'Kabupaten Bondowoso', 37: 'Kabupaten Situbondo', 
    38: 'Kabupaten Banyuwangi'
}
df['nama_wilayah'] = df['nama_wilayah'].astype(int).map(pemetaan_id_kota)
df.dropna(subset=['nama_wilayah'], inplace=True) 

print("Melakukan Resampling Per Jam dan Membersihkan Anomali Sensor...")
df['waktu_aktual'] = pd.to_datetime(df['waktu_aktual']).dt.tz_localize(None)
df.set_index('waktu_aktual', inplace=True)

df_hourly = df.groupby('nama_wilayah')[daftar_polutan].resample('H').mean().reset_index()

# Menghancurkan anomali mutlak (0 dan error sensor -9999)
df_hourly[daftar_polutan] = df_hourly[daftar_polutan].replace({-9999.0: np.nan, 0.0: np.nan})

print("Menambal data kosong dengan Interpolasi Linier...")
df = df_hourly.groupby('nama_wilayah', group_keys=False).apply(
    lambda group: group.interpolate(method='linear')
).reset_index(drop=True)

df = df.bfill().ffill()
print(f"Data bersih dan siap! Dimensi: {df.shape}")

Membuka jembatan ke Supabase...
Menyedot batch ke-1...
Menyedot batch ke-2...
Menyedot batch ke-3...
Menyedot batch ke-4...
Menyedot batch ke-5...
Menyedot batch ke-6...
Menyedot batch ke-7...
Menyedot batch ke-8...
Menyedot batch ke-9...
Menyedot batch ke-10...
Menyedot batch ke-11...
Menyedot batch ke-12...
Menyedot batch ke-13...
Menyedot batch ke-14...
Melakukan Resampling Per Jam dan Membersihkan Anomali Sensor...
Menambal data kosong dengan Interpolasi Linier...
Data bersih dan siap! Dimensi: (665760, 8)


In [ ]:
kolom_waktu = 'waktu_aktual'
kolom_kota = 'nama_wilayah'

def racik_fitur_deret_waktu(df_kota, n_lags=24, n_targets=24):
    df_temp = df_kota.sort_values(kolom_waktu).copy()
    
    # Fitur Temporal
    df_temp['Bulan'] = df_temp[kolom_waktu].dt.month
    df_temp['Jam'] = df_temp[kolom_waktu].dt.hour
    df_temp['Is_Weekend'] = df_temp[kolom_waktu].dt.dayofweek.isin([5, 6]).astype(int)

    for p in daftar_polutan:
        # Fitur Sejarah Murni (Lag)
        for i in range(1, n_lags + 1):
            df_temp[f'{p}_H_{i}'] = df_temp[p].shift(i)
            
        # Fitur Tren Panjang
        df_temp[f'{p}_Mean_72'] = df_temp[p].rolling(window=72, min_periods=1).mean()
        df_temp[f'{p}_Max_72'] = df_temp[p].rolling(window=72, min_periods=1).max()
        
        # Target Masa Depan (T+1 hingga T+24)
        for h in range(1, n_targets + 1):
            df_temp[f'TARGET_{p}_T_{h}'] = df_temp[p].shift(-h)
            
    return df_temp

tqdm.pandas(desc="Meracik Fitur 24H per Kota")
df_model = df.groupby(kolom_kota, group_keys=False).progress_apply(racik_fitur_deret_waktu).reset_index(drop=True)

kolom_target = [col for col in df_model.columns if 'TARGET_' in col]
kolom_fitur = [col for col in df_model.columns if col not in kolom_target and col not in [kolom_waktu]]

print("Membersihkan baris tanpa target masa depan...")
df_model = df_model.dropna(subset=kolom_target).reset_index(drop=True)
df_model[kolom_fitur] = df_model[kolom_fitur].fillna(0)

# Simpan sebagai Checkpoint Parquet (Teknik Big Data)
nama_file_checkpoint = "dataset_engineered_checkpoint.parquet"
df_model.to_parquet(nama_file_checkpoint, index=False)
print(f"Checkpoint matriks berhasil disimpan di {nama_file_checkpoint}!")

del df, df_hourly
gc.collect()

Meracik Fitur 24H per Kota:   0%|          | 0/38 [00:00<?, ?it/s]

Membersihkan baris tanpa target masa depan...
Checkpoint matriks berhasil disimpan di dataset_engineered_checkpoint.parquet!


1008

In [ ]:
print("Membaca data dari Checkpoint Parquet dan menerapkan Diet Memori...")
df_model = pd.read_parquet(nama_file_checkpoint)

# One-Hot Encoding Lokasi
df_model[kolom_kota] = df_model[kolom_kota].astype(str)
df_model = pd.get_dummies(df_model, columns=[kolom_kota], dtype='float32')

# Urutkan berdasarkan waktu mutlak agar TimeSeriesSplit bekerja optimal
df_model = df_model.sort_values(kolom_waktu).reset_index(drop=True)

y = df_model[kolom_target].astype('float32')
X = df_model.drop(columns=kolom_target + [kolom_waktu], errors='ignore').astype('float32')

del df_model
gc.collect()

print("Membelah data menjadi Train & Test...")
X_train, X_test, Y_train, Y_test = train_test_split(X.values, y.values, test_size=0.2, shuffle=False)

print("Menerapkan Transformasi Logaritmik (Log1p)...")
Y_train_log = np.log1p(Y_train)

# Memastikan array C-Contiguous untuk XGBoost
X_train = np.ascontiguousarray(X_train)
X_test = np.ascontiguousarray(X_test)
Y_train_log = np.ascontiguousarray(Y_train_log)
Y_test = np.ascontiguousarray(Y_test)

del X, y
gc.collect()
print(f"Kesiapan Memori: X_train {X_train.shape} | Y_train_log {Y_train_log.shape}")

Membaca data dari Checkpoint Parquet dan menerapkan Diet Memori...
Membelah data menjadi Train & Test...
Menerapkan Transformasi Logaritmik (Log1p)...
Kesiapan Memori: X_train (531878, 203) | Y_train_log (531878, 144)


In [ ]:
print("Memulai Optuna Ekstrem dengan TimeSeriesSplit & Checkpointing SQLite...")

kelompok_target = {}
for polutan in daftar_polutan:
    idx_kolom = [i for i, col in enumerate(kolom_target) if polutan in col.upper()]
    kelompok_target[polutan] = idx_kolom

model_terbaik_per_polutan = {}

# Fungsi Objektif yang menerapkan Validasi Silang Kronologis (Mencegah Overfitting)
def objective(trial, X_ctx, y_ctx):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'random_state': 42,
        'tree_method': "hist",
        'device': "cuda"
        # multi_strategy dihapus, kita menggunakan MultiOutputRegressor asli
    }
    
    tscv = TimeSeriesSplit(n_splits=3)
    skor_fold = []
    
    # Pengujian berjalan maju layaknya dunia nyata
    for train_idx, val_idx in tscv.split(X_ctx):
        X_tr, y_tr = X_ctx[train_idx], y_ctx[train_idx]
        X_va, y_va = X_ctx[val_idx], y_ctx[val_idx]
        
        # PENGGUNAAN MULTIOUTPUTREGRESSOR YANG BENAR (n_jobs=1 agar VRAM aman)
        xgb_base = XGBRegressor(**params)
        model = MultiOutputRegressor(xgb_base, n_jobs=1)
        model.fit(X_tr, y_tr)
        
        preds_log = model.predict(X_va)
        
        # PERBAIKAN BUG SCIKIT-LEARN VERSI TERBARU
        rmse_fold = np.sqrt(mean_squared_error(y_va, preds_log))
        skor_fold.append(rmse_fold)
        
        del X_tr, y_tr, X_va, y_va, model
        gc.collect()
        
    return np.mean(skor_fold)

# Checkpointing SQLite: Jika Kaggle mati, proses Optuna akan berlanjut, bukan diulang!
nama_db_optuna = "sqlite:///optuna_aero_checkpoint.db"

with mlflow.start_run(run_name="Master_TimeSeries_GPU"):
    for nama_polutan, indeks_kolom in kelompok_target.items():
        print(f"\n=========================================")
        print(f"Mendidik Model MultiOutput Asli: {nama_polutan}")
        print(f"=========================================")
        
        y_train_chunk = Y_train_log[:, indeks_kolom]
        
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        # Menghubungkan studi ke SQLite
        study = optuna.create_study(study_name=f"study_{nama_polutan}", 
                                    storage=nama_db_optuna, 
                                    load_if_exists=True, 
                                    direction="minimize")
        
        # Eksekusi Mentok Kanan
        study.optimize(lambda trial: objective(trial, X_train, y_train_chunk), 
                       n_trials=10, show_progress_bar=True) 
        
        best_params = study.best_params
        print(f"Kombinasi Terbaik {nama_polutan}: {best_params}")
        
        print("Melatih bobot final dengan 100% data Train...")
        
        # PENGGUNAAN MULTIOUTPUTREGRESSOR UNTUK MODEL FINAL
        best_xgb = XGBRegressor(**best_params, random_state=42, tree_method="hist", device="cuda")
        final_model_chunk = MultiOutputRegressor(best_xgb, n_jobs=1)
        final_model_chunk.fit(X_train, y_train_chunk)
        
        model_terbaik_per_polutan[nama_polutan] = final_model_chunk
        mlflow.log_params({f"{nama_polutan}_{k}": v for k, v in best_params.items()})
        gc.collect()

print("\nSeluruh tahapan pencarian Hyperparameter Ekstrem selesai!")

Memulai Optuna Ekstrem dengan TimeSeriesSplit & Checkpointing SQLite...

Mendidik Model MultiOutput Asli: PM25


  0%|          | 0/10 [00:00<?, ?it/s]

Kombinasi Terbaik PM25: {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.01790873997058966, 'subsample': 0.6923099973742468, 'colsample_bytree': 0.629688400138903, 'min_child_weight': 5, 'gamma': 2.874242043401086, 'reg_alpha': 0.019834304498906177, 'reg_lambda': 0.0004727441397574371}
Melatih bobot final dengan 100% data Train...

Mendidik Model MultiOutput Asli: PM10


  0%|          | 0/10 [00:00<?, ?it/s]

Kombinasi Terbaik PM10: {'n_estimators': 250, 'max_depth': 3, 'learning_rate': 0.034967377425417744, 'subsample': 0.8263376654794836, 'colsample_bytree': 0.6212863159157215, 'min_child_weight': 9, 'gamma': 1.0285111961563964, 'reg_alpha': 0.7494746975241156, 'reg_lambda': 0.15816745733504597}
Melatih bobot final dengan 100% data Train...

Mendidik Model MultiOutput Asli: SO2


...

Kombinasi Terbaik CO: {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.030767848074975977, 'subsample': 0.6521794053319845, 'colsample_bytree': 0.8837327910385361, 'min_child_weight': 2, 'gamma': 0.24664680327491362, 'reg_alpha': 0.016792725757866837, 'reg_lambda': 0.06313993714589743}
Melatih bobot final dengan 100% data Train...

Mendidik Model MultiOutput Asli: NO2


  0%|          | 0/10 [00:00<?, ?it/s]

Kombinasi Terbaik NO2: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.03698521164985141, 'subsample': 0.821175465960454, 'colsample_bytree': 0.893041867126392, 'min_child_weight': 7, 'gamma': 1.409230308821217, 'reg_alpha': 0.0022075795422405453, 'reg_lambda': 0.36493040935232474}
Melatih bobot final dengan 100% data Train...

Mendidik Model MultiOutput Asli: O3


  0%|          | 0/10 [00:00<?, ?it/s]

Kombinasi Terbaik O3: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.016629436537073616, 'subsample': 0.7540038576930177, 'colsample_bytree': 0.726191100088302, 'min_child_weight': 1, 'gamma': 4.495325270766221, 'reg_alpha': 0.0003877285853072599, 'reg_lambda': 0.0006776915126567988}
Melatih bobot final dengan 100% data Train...
🏃 View run Master_TimeSeries_GPU at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/7/runs/8b2243e3a5514bca8345f0f8529352fc
🧪 View experiment at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/7

Seluruh tahapan pencarian Hyperparameter Ekstrem selesai!


In [ ]:
print("Merakit kembali prediksi ke bentuk aslinya...")

Y_pred_asli_gabungan = np.empty_like(Y_test)

for nama_polutan, indeks_kolom in kelompok_target.items():
    model_spesifik = model_terbaik_per_polutan[nama_polutan]
    pred_log_spesifik = model_spesifik.predict(X_test)
    Y_pred_asli_gabungan[:, indeks_kolom] = np.expm1(pred_log_spesifik)

WINDOW_OUTPUT = 24
Y_test_reshaped = Y_test.reshape(-1, len(daftar_polutan), WINDOW_OUTPUT)
Y_pred_reshaped = Y_pred_asli_gabungan.reshape(-1, len(daftar_polutan), WINDOW_OUTPUT)

print("--- TABEL STATISTIK AKURASI TIAP POLUTAN (MULTI-OUTPUT) ---")
statistik_polutan = []

for idx, polutan in enumerate(daftar_polutan):
    # Dimensi sekarang: (Sampel, Polutan, Horizon Jam)
    y_true = Y_test_reshaped[:, idx, :]
    y_pred = Y_pred_reshaped[:, idx, :]
    
    rata_rata_asli = y_true.mean()
    mae = mean_absolute_error(y_true, y_pred)
    
    # PERBAIKAN BUG SCIKIT-LEARN VERSI TERBARU
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    persentase_error = (mae / rata_rata_asli) * 100 if rata_rata_asli != 0 else 0
    
    statistik_polutan.append({
        "Polutan": polutan, "Rata-rata (µg/m³)": f"{rata_rata_asli:.2f}",
        "MAE": f"{mae:.2f}", "RMSE": f"{rmse:.2f}", "R2 Score": f"{r2:.4f}", "Error (%)": f"{persentase_error:.2f}%"
    })

display(pd.DataFrame(statistik_polutan))

print("\nMemulai proses pengamanan model...")
nama_file_model = "aeroguard_timeseries_multioutput.pkl"

paket_model = {
    'dict_model_spesialis': model_terbaik_per_polutan,
    'fitur': list(df_model.columns) if 'df_model' in locals() else [] 
}

joblib.dump(paket_model, nama_file_model)

with mlflow.start_run(run_name="Master_TimeSeries_GPU", nested=True):
    mlflow.log_artifact(nama_file_model)

print("Mengirim paket model ke Google Drive...")
try:
    file_drive = drive.CreateFile({'title': nama_file_model})
    file_drive.SetContentFile(nama_file_model)
    file_drive.Upload()
    print(f"BAM! File '{nama_file_model}' berhasil diamankan ke Google Drive.")
except Exception as e:
    print(f"[WARN] Pengiriman ke GDrive terhambat: {e}. File tetap aman di DagsHub!")

print("✅ Seluruh siklus MLOps malam ini telah selesai. Selamat beristirahat!")

Merakit kembali prediksi ke bentuk aslinya...
--- TABEL STATISTIK AKURASI TIAP POLUTAN (MULTI-OUTPUT) ---


,Polutan,Rata-rata (µg/m³),MAE,RMSE,R2 Score,Error (%)
0,PM25,8.87,3.99,8.40,0.5864,44.94%
1,PM10,12.50,4.94,9.31,0.6132,39.51%
2,SO2,0.71,0.29,0.59,0.7353,40.82%
3,CO,221.50,60.76,107.48,0.7566,27.43%
4,NO2,2.01,0.90,1.86,0.7295,44.67%
5,O3,38.10,8.61,14.98,0.4526,22.60%



Memulai proses pengamanan model...
🏃 View run Master_TimeSeries_GPU at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/7/runs/725ee9dc06bf4d9baee463428c034ef2
🧪 View experiment at: https://dagshub.com/tegarkusuma12/Web-ISPU.mlflow/#/experiments/7
Mengirim paket model ke Google Drive...
BAM! File 'aeroguard_timeseries_multioutput.pkl' berhasil diamankan ke Google Drive.
✅ Seluruh siklus MLOps malam ini telah selesai. Selamat beristirahat!
